# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs. List their structure and fields using their `@id` references as required in FAIR data workflows.

In [ ]:
# List all RecordSets and their fields by `@id`
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for recset in record_sets:
    print(f"RecordSet name: {recset.name} (@id: {recset.id})")
    record_set_ids.append(recset.id)
    print("  Fields and columns:")
    for field in recset.fields:
        print(f"    Field: {field.name} (@id: {field.id}) - dataType: {field.data_type}")
        # List columns if available
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"      Column: {getattr(column, 'name', None)} (@id: {getattr(column, 'id', None)})")
    print("")

## Example records from one record set
Let's show example records by iterating records for the first listed record set. All entities are referenced by their `@id` fields.

In [ ]:
if record_sets:
    example_recset_id = record_sets[0].id
    print(f"--- Example records for record set '@id': {example_recset_id} ---")
    cnt = 0
    for rec in dataset.records(record_set=example_recset_id):
        print(json.dumps(rec, indent=2)[:500])
        cnt += 1
        if cnt >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set using their `@id`
dataframes = {}
# Recompute record_set_ids if restarting cell
record_set_ids = [rset.id for rset in dataset.record_sets()]
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records_list = list(records_iter)
    if records_list:
        df = pd.DataFrame(records_list)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {record_set_id}")
    else:
        print(f"No records found for RecordSet '@id': {record_set_id}")
# Display the columns of the first DataFrame:
if dataframes:
    first_recset_id = list(dataframes.keys())[0]
    print(f"DataFrame columns for '@id': {first_recset_id}")
    print(dataframes[first_recset_id].columns.tolist())
    display(dataframes[first_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalizing, and simple grouping. Operations reference the field IDs for full reproducibility.

In [ ]:
# Pick first record set with rows and at least one numeric field
from collections.abc import Iterable
numeric_field_id = None
group_field_id = None
ed_recset_id = None
for recset in dataset.record_sets():
    df = dataframes.get(recset.id, None)
    if df is not None and not df.empty:
        # Find first numeric field (float/int)
        numeric_fields = [f for f in recset.fields if f.data_type in ('Number','Float','Integer','schema:Number','schema:Float','schema:Integer')]
        if numeric_fields:
            numeric_field_id = numeric_fields[0].id
            # Attempt to find a group-able (Text) field
            groupable_fields = [f for f in recset.fields if f.data_type in ('Text','schema:Text','String','schema:String')]
            if groupable_fields:
                group_field_id = groupable_fields[0].id
            ed_recset_id = recset.id
            break
# If none found, fallback to any float/int-like column

print(f"Using record set '@id': {ed_recset_id}")
print(f"Numeric field selected by @id: {numeric_field_id}")
print(f"Group field selected by @id: {group_field_id}")

if ed_recset_id and numeric_field_id in dataframes[ed_recset_id].columns:
    threshold = dataframes[ed_recset_id][numeric_field_id].mean() if pd.api.types.is_numeric_dtype(dataframes[ed_recset_id][numeric_field_id]) else 10
    filtered_df = dataframes[ed_recset_id][dataframes[ed_recset_id][numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:0.3f}:")
    display(filtered_df.head())
    # Normalize
    normcol = f"{numeric_field_id}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normcol]].head())
    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"Grouped data by {group_field_id} (mean value of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No numeric field found suitable for EDA in available record sets.')

## 5. Visualization
Visualize a numeric field's distribution and (if available) how it varies by a group field, using the referenced `@id` field names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if ed_recset_id and numeric_field_id in dataframes[ed_recset_id].columns:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[ed_recset_id][numeric_field_id].astype(float), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in record set {ed_recset_id}")
    plt.show()
    # Boxplot by group_field
    if group_field_id and group_field_id in dataframes[ed_recset_id].columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[ed_recset_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
This notebook demonstrated stepwise exploration of the FAIR² dataset using the `mlcroissant` library. Data entities were referenced by their `@id` for reproducible workflows. For full dataset documentation, interpretation of field meanings, and advanced analyses, see the [original study](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the Croissant schema.